In [ ]:
import re
import json
import warnings
import traceback
import importlib
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import f_classif

from pydantic import BaseModel, Field, model_validator, ValidationError

# reducir ruido en las pruebas
warnings.filterwarnings("ignore")  

# Robust ML Pipeline Parsing, Construction & Evaluation from LLM Responses

Este notebook implementa un flujo de trabajo para:
1. **Extraer** todos los posibles pipelines JSON que aparezcan en una respuesta de un LLM.
2. **Validar** la estructura y las restricciones de hiperparámetros usando `sklearn_map.json` mediante modelos de Pydantic, generando errores detallados si algo falla.
3. **Construir** pipelines de Scikit‑Learn válidos a partir de las especificaciones correctas.
4. **Evaluar** el pipeline con validación cruzada, devolviendo mensajes de error concisos y orientados a la causa raíz.

Se incluyen múltiples casos de prueba que simulan respuestas típicas (y atípicas) de un LLM.

## Carga de `sklearn_map.json` y Construcción de Espacio de Búsqueda de Componentes Permitidos

El resultado es un diccionario `ALLOWED_COMPONENTS` que contiene para cada componente: clase importada, stage, hiperparámetros con sus restricciones (tipo, rango, valores permitidos, nullable, etc.).

In [ ]:
PATH = "../src/sklearn_map.json"

def load_sklearn_map(path: str = "sklearn_map.json") -> Dict:
  "Carga el archivo JSON que define los componentes e hiperparámetros permitidos."
  with open(PATH, "r", encoding="utf-8") as f:
    return json.load(f)
  raise Exception(f"Ha ocurrido un problema al cargar el archivo JSON: {path}")

def build_allowed_components(sklearn_map: Dict) -> Dict[str, Dict]:
  "Construye el diccionario ALLOWED_COMPONENTS a partir del mapa cargado."
  ALLOWED = {}

  stage_map = {
      "imputer": "preprocessor",
      "scaler": "preprocessor",
      "encoder": "preprocessor",
      "transformer": "preprocessor",
      "feature_selection": "feature_selection",
      "dimensionality_reduction": "feature_selection",
      "column_transformer": "preprocessor",
  }

  # Preprocesamientos
  for step_def in sklearn_map.get("preprocessing_steps", []):
    name = step_def["name"]
    module_path = step_def["module"]
    # Importar la clase
    mod_name, cls_name = module_path.rsplit(".", 1)
    try:
      mod = importlib.import_module(mod_name)
      cls = getattr(mod, cls_name)
    except (ImportError, AttributeError) as e:
      raise RuntimeError(f"No se pudo importar {module_path}: {e}")

    hyperparams = {}
    for hp_name, hp_def in step_def.get("hyperparameters", {}).items():
      hp_type = hp_def["type"]
      search_space = hp_def.get("search_space", [])
      nullable = None in search_space if search_space else False

      if hp_type == "categorical":
        # Si todos los no nulos son bool, consideramos booleano
        non_null = [v for v in search_space if v is not None]
        if all(isinstance(v, bool) for v in non_null):
          param_type = bool
          allowed_vals = [v for v in search_space if v is not None]
        else:
          param_type = object  # acepta cualquier valor dentro de search_space
          allowed_vals = search_space
        hyperparams[hp_name] = {
            "type": param_type,
            "allowed": allowed_vals,
            "nullable": nullable,
        }
      elif hp_type in ("integer", "continuous"):
        py_type = int if hp_type == "integer" else float
        rng = search_space  # [min, max]
        hyperparams[hp_name] = {
            "type": py_type,
            "range": {"min": rng[0], "max": rng[1]},
            "nullable": nullable,
        }
      elif hp_type == "list":
        # No validamos estructura interna; aceptamos cualquier objeto
        hyperparams[hp_name] = {"type": object, "nullable": nullable}
      else:
        hyperparams[hp_name] = {"type": object, "nullable": nullable}

    stage = stage_map.get(step_def.get("step", ""), "preprocessor")
    ALLOWED[name] = {
        "class": cls,
        "stage": stage,
        "hyperparameters": hyperparams,
    }

  # Clasificadores
  for clf_def in sklearn_map.get("sklearn_classification_algorithms", []):
    name = clf_def["name"]
    module_path = clf_def["module"]
    mod_name, cls_name = module_path.rsplit(".", 1)
    mod = importlib.import_module(mod_name)
    cls = getattr(mod, cls_name)

    hyperparams = {}
    for hp_name, hp_def in clf_def.get("hyperparameters", {}).items():
      hp_type = hp_def["type"]
      search_space = hp_def.get("search_space", [])
      nullable = None in search_space if search_space else False

      if hp_type == "categorical":
        non_null = [v for v in search_space if v is not None]
        if all(isinstance(v, bool) for v in non_null):
          param_type = bool
          allowed_vals = [v for v in search_space if v is not None]
        else:
          param_type = object
          allowed_vals = search_space
        hyperparams[hp_name] = {
            "type": param_type,
            "allowed": allowed_vals,
            "nullable": nullable,
        }
      elif hp_type in ("integer", "continuous"):
        py_type = int if hp_type == "integer" else float
        rng = search_space
        hyperparams[hp_name] = {
            "type": py_type,
            "range": {"min": rng[0], "max": rng[1]},
            "nullable": nullable,
        }
      else:
        hyperparams[hp_name] = {"type": object, "nullable": nullable}

    ALLOWED[name] = {
        "class": cls,
        "stage": "classifier",
        "hyperparameters": hyperparams,
    }

  # Parches fijos: score_func para SelectKBest/SelectPercentile
  for name in ["SelectKBest", "SelectPercentile"]:
    if name in ALLOWED:
      ALLOWED[name]["fixed_params"] = {"score_func": f_classif}

  return ALLOWED

# Carga única del mapa
sklearn_map = load_sklearn_map()
ALLOWED_COMPONENTS = build_allowed_components(sklearn_map)

## Definición de modelos Pydantic para la especificación del Pipeline

In [ ]:
# Auxiliar Functions
def _validate_column_transformer_transformers(transformers_value: List) -> List[str]:
  "Valida la estructura de la lista 'transformers' de ColumnTransformer. Retornando una lista de mensajes de error (vacía si todo correcto)"
  errors = []
  if not isinstance(transformers_value, list):
    return ["'transformers' debe ser una lista."]

  # Claves permitidas en cada objeto transformador
  ALLOWED_KEYS = {"name", "transformer", "columns", "transformer_hyperparameters"}

  for idx, item in enumerate(transformers_value):
    prefix = f"[{idx}]"

    # El elemento debe ser un diccionario, no una lista
    if not isinstance(item, dict):
      errors.append(f"{prefix}: se esperaba un objeto (dict), pero se recibió {type(item).__name__}.")
      continue

    # Detectar claves no permitidas y sugerir el nombre correcto si ponen 'hyperparameters'
    extra_keys = set(item.keys()) - ALLOWED_KEYS
    if extra_keys:
      if "hyperparameters" in extra_keys:
        errors.append(f"{prefix}: clave no permitida 'hyperparameters'. Use 'transformer_hyperparameters' para los parámetros del sub-transformador.")
        extra_keys.discard("hyperparameters")
      if extra_keys:
        errors.append(f"{prefix}: claves no reconocidas: {extra_keys}. Claves permitidas: {ALLOWED_KEYS}.")
      # Si hay claves extra, continuamos para poder reportar todos los errores,
      # pero no seguimos validando el contenido de las claves obligatorias que puedan faltar.

    # Claves obligatorias
    missing = []
    for key in ("name", "transformer", "columns"):
      if key not in item:
        missing.append(key)
    if missing:
      errors.append(f"{prefix}: faltan las claves obligatorias: {missing}.")
      continue

    # Validar 'name'
    name = item.get("name")
    if not isinstance(name, str):
      errors.append(f"{prefix}.name: debe ser una cadena de texto (str).")

    # Validar 'transformer'
    transformer = item.get("transformer")
    if transformer is None:
      errors.append(f"{prefix}.transformer: es obligatorio y no puede ser None.")
    elif not isinstance(transformer, str):
      errors.append(f"{prefix}.transformer: debe ser un nombre de componente (str), no {type(transformer).__name__}.")
    elif transformer not in ALLOWED_COMPONENTS:
      errors.append(f"{prefix}.transformer '{transformer}': no es un componente permitido. Componentes válidos: {list(ALLOWED_COMPONENTS.keys())}.")
    else:
      # Validar hiperparámetros del sub-transformador, si existen
      sub_hp = item.get("transformer_hyperparameters", {})
      if not isinstance(sub_hp, dict):
        errors.append(f"{prefix}.transformer_hyperparameters: debe ser un objeto (dict).")
      else:
        sub_schema = ALLOWED_COMPONENTS[transformer]["hyperparameters"]
        allowed_hp = set(sub_schema.keys())
        given_hp = set(sub_hp.keys())
        unknown = given_hp - allowed_hp
        if unknown:
          errors.append(f"{prefix} '{transformer}': hiperparámetros desconocidos {unknown}. Válidos: {allowed_hp if allowed_hp else '(ninguno)'}.")
        # Validación individual de cada hiperparámetro (tipo, rango, etc.)
        for hp_name, hp_value in sub_hp.items():
          if hp_name not in sub_schema:
            continue
          hp_schema = sub_schema[hp_name]
          expected_type = hp_schema["type"]
          nullable = hp_schema.get("nullable", False)
          if hp_value is None:
            if not nullable:
              errors.append(f"{prefix}.{transformer}.{hp_name}: no acepta None.")
            continue
          # Coerción simple
          coerced = hp_value
          if expected_type == float and isinstance(hp_value, int):
            coerced = float(hp_value)
          elif expected_type == int and isinstance(hp_value, float) and hp_value.is_integer():
            coerced = int(hp_value)
          elif expected_type == bool and isinstance(hp_value, str):
            if hp_value.lower() in ("true", "false"):
              coerced = hp_value.lower() == "true"
            else:
              errors.append(f"{prefix}.{transformer}.{hp_name}: se esperaba bool, pero la cadena '{hp_value}' no es válida.")
              continue
          if not isinstance(coerced, expected_type):
            errors.append(f"{prefix}.{transformer}.{hp_name}: tipo incorrecto. Se esperaba {expected_type.__name__}, se recibió {type(hp_value).__name__}.")
            continue
          if "allowed" in hp_schema and coerced not in hp_schema["allowed"]:
            errors.append(f"{prefix}.{transformer}.{hp_name}: valor '{coerced}' no permitido. Valores válidos: {hp_schema['allowed']}.")
          if "range" in hp_schema:
            rng = hp_schema["range"]
            if not (rng["min"] <= coerced <= rng["max"]):
              errors.append(f"{prefix}.{transformer}.{hp_name}: valor {coerced} fuera de rango. Rango permitido: [{rng['min']}, {rng['max']}].")

    # Validar 'columns'
    columns = item.get("columns")
    if not isinstance(columns, list):
      errors.append(f"{prefix}.columns: debe ser una lista de índices o nombres, no {type(columns).__name__}.")
    else:
      for col_idx, col in enumerate(columns):
        if not isinstance(col, (int, str)):
          errors.append(f"{prefix}.columns[{col_idx}]: debe ser int o str, no {type(col).__name__}.")

  return errors

class StepSpec(BaseModel):
  name: str
  component: str
  hyperparameters: Dict[str, Any] = Field(default_factory=dict)

  @model_validator(mode="after")
  def validate_component_and_hyperparams(self):
    comp = self.component
    if comp not in ALLOWED_COMPONENTS:
      allowed = list(ALLOWED_COMPONENTS.keys())
      raise ValueError(f"Componente '{comp}' no permitido. Permitidos: {allowed}")

    schema = ALLOWED_COMPONENTS[comp]
    allowed_hp = set(schema["hyperparameters"].keys())
    given_hp = set(self.hyperparameters.keys())
    extra = given_hp - allowed_hp
    if extra:
      raise ValueError(f"Hiperparámetros desconocidos para '{comp}': {extra}. Válidos: {allowed_hp if allowed_hp else '(ninguno)'}")

    if self.component == "ColumnTransformer" and "transformers" in self.hyperparameters:
      transformers = self.hyperparameters["transformers"]
      ct_errors = _validate_column_transformer_transformers(transformers)
      if ct_errors:
        raise ValueError(f"La lista 'transformers' de ColumnTransformer no es válida: {'; '.join(ct_errors)}")

    # Validación individual
    for hp_name, hp_val in self.hyperparameters.items():
      if hp_name not in schema["hyperparameters"]:
        continue  # ya se reportó como extra antes
      hp_schema = schema["hyperparameters"][hp_name]
      nullable = hp_schema.get("nullable", False)

      if hp_val is None:
        if not nullable:
          raise ValueError(f"'{comp}.{hp_name}' no puede ser None.")
        continue

      expected_type = hp_schema["type"]
      # Coerción sencilla para ayudar
      coerced = hp_val
      if expected_type == float and isinstance(hp_val, int):
        coerced = float(hp_val)
      elif expected_type == int and isinstance(hp_val, float) and hp_val.is_integer():
        coerced = int(hp_val)
      elif expected_type == bool and isinstance(hp_val, str):
        if hp_val.lower() in ("true", "false"):
          coerced = hp_val.lower() == "true"
        else:
          raise ValueError(f"'{comp}.{hp_name}': se esperaba bool, pero la cadena '{hp_val}' no es 'true' ni 'false'.")
      elif expected_type == tuple and isinstance(hp_val, list):
        coerced = tuple(hp_val)
      elif expected_type == object:
        # no coercion, aceptamos cualquier cosa (ej. listas)
        pass
      else:
        if not isinstance(coerced, expected_type):
          raise ValueError(f"'{comp}.{hp_name}': tipo inválido. Se esperaba {expected_type.__name__}, se encontró {type(hp_val).__name__}.")

      # Validación de rango/allowed
      if "allowed" in hp_schema:
        if coerced not in hp_schema["allowed"]:
          raise ValueError(f"'{comp}.{hp_name}': valor '{coerced}' no permitido. Valores válidos: {hp_schema['allowed']}")
      if "range" in hp_schema:
        rng = hp_schema["range"]
        if not (rng["min"] <= coerced <= rng["max"]):
          raise ValueError(f"'{comp}.{hp_name}': valor {coerced} fuera de rango. Rango permitido: [{rng['min']}, {rng['max']}].")
    return self

class PipelineSpec(BaseModel):
  steps: List[StepSpec]

  @model_validator(mode="after")
  def check_at_least_one_step(self):
    if len(self.steps) == 0:
      raise ValueError("El pipeline debe tener al menos un paso.")
    return self


## Extracción de candidatos JSON desde una respuesta de texto

In [ ]:
def extract_json_candidates(text: str) -> List[dict]:
  "Extrae todos los objetos JSON (diccionarios) completos del texto. Soporta bloques markdown ```json ... ``` y JSON suelto, incluso si hay fragmentos malformados intercalados"
  candidates = []
  # 1. Extraer de bloques markdown (```json ... ```)
  md_pattern = re.compile(r"```(?:json)?\s*(\{.*?\})\s*```", re.DOTALL)
  for match in md_pattern.finditer(text):
    try:
      obj = json.loads(match.group(1))
      candidates.append(obj)
    except json.JSONDecodeError:
      pass

  # 2. Extraer objetos JSON fuera de bloques markdown, usando raw_decode
  #    que maneja correctamente fragmentos no válidos sin afectar a los posteriores.
  #    Primero eliminamos del texto las partes que ya capturamos con markdown
  #    para no duplicar.
  cleaned_text = text
  for match in md_pattern.finditer(text):
    cleaned_text = cleaned_text.replace(match.group(0), " ")  # reemplazar por espacios

  decoder = json.JSONDecoder()
  idx = 0
  while idx < len(cleaned_text):
    # Buscar el siguiente '{'
    brace_idx = cleaned_text.find('{', idx)
    if brace_idx == -1:
      break
    try:
      # Intentar decodificar un objeto JSON completo desde esa posición
      obj, end_idx = decoder.raw_decode(cleaned_text, brace_idx)
      candidates.append(obj)
      idx = end_idx  # avanzar justo después del objeto extraído
    except (json.JSONDecodeError, ValueError) as e:
      # No se pudo decodificar un JSON válido desde aquí.
      # Saltamos el carácter problemático y seguimos buscando.
      idx = brace_idx + 1

  return candidates


## Funciones de Construcción y Evaluación del Pipeline de Scikit-Learn

In [ ]:
def _build_sklearn_step(step_spec: StepSpec) -> Tuple[str, Any]:
  "Construye una tupla (nombre, estimador) para un paso individual."
  comp_name = step_spec.component
  schema = ALLOWED_COMPONENTS[comp_name]
  raw_params = step_spec.hyperparameters.copy()

  # construcción para el caso especial de ColumnTransformer
  if comp_name == "ColumnTransformer":
    transformers_list = raw_params.get("transformers", [])
    built_transformers = []
    for tf_dict in transformers_list:
      name = tf_dict["name"]
      sub_comp = tf_dict["transformer"]
      if sub_comp in ("drop", "passthrough"):
        transformer_obj = sub_comp
      else:
        sub_hp = tf_dict.get("transformer_hyperparameters", {})
        sub_step_spec = StepSpec(name=name, component=sub_comp, hyperparameters=sub_hp)
        _, transformer_obj = _build_sklearn_step(sub_step_spec)
      columns = tf_dict["columns"]
      built_transformers.append((name, transformer_obj, columns))
    raw_params["transformers"] = built_transformers

  # Coerción de tipos
  coerced = {}
  for k, v in raw_params.items():
    if k == "transformers" and comp_name == "ColumnTransformer":
      coerced[k] = v
      continue
    # Si el hiperparámetro no está en el esquema, lo pasamos tal cual (no debería ocurrir si se validó)
    if k not in schema["hyperparameters"]:
      coerced[k] = v
      continue
    expected = schema["hyperparameters"][k]["type"]
    # Conversiones análogas a las usadas en validación
    if expected == float and isinstance(v, int):
      coerced[k] = float(v)
    elif expected == int and isinstance(v, float) and v == int(v):
      coerced[k] = int(v)
    elif expected == bool and isinstance(v, str):
      coerced[k] = v.lower() == "true"
    elif expected == tuple and isinstance(v, list):
      coerced[k] = tuple(v)
    else:
      coerced[k] = v

  # Parche específico para MinMaxScaler.feature_range
  if comp_name == "MinMaxScaler" and "feature_range" in coerced:
    fr = coerced["feature_range"]
    if isinstance(fr, list) and len(fr) == 2:
      coerced["feature_range"] = tuple(fr)

  # Parámetros fijos como score_func
  fixed = schema.get("fixed_params", {})
  coerced.update(fixed)

  estimator = schema["class"](**coerced)
  return step_spec.name, estimator

def build_pipeline(spec: PipelineSpec) -> Pipeline:
  "Construye un sklearn Pipeline a partir de una especificación válidada. Lanza excepciones propias de scikit-learn o de Python si los parámetros no son aceptados por el constructor"
  steps = []
  for step_spec in spec.steps:
    name, est = _build_sklearn_step(step_spec)
    steps.append((name, est))
  return Pipeline(steps)

def _extract_core_error(exc: Exception) -> str:
  "Reduce un traceback complejo a solo la línea más informativa."
  msg = str(exc)
  # Caso típico: "All the X fits failed." contiene líneas con la causa real
  if "All the" in msg and "fits failed" in msg:
    lines = msg.splitlines()
    for line in lines:
      for err_type in ("ValueError:", "TypeError:", "KeyError:", "IndexError:"):
        if err_type in line:
          return line.strip()
    # Si no encontramos, devolvemos la última línea que suele contener el motivo
    return lines[-1].strip() if lines else msg[:500]
  # Otras excepciones: limitamos longitud
  if len(msg) > 500:
    return msg[:250] + "..." + msg[-250:]
  return msg


@dataclass
class EvaluationResult:
  success: bool
  metrics: Dict[str, float] = field(default_factory=dict)
  errors: List[str] = field(default_factory=list)
  warnings: List[str] = field(default_factory=list)

  def to_feedback(self, add_warning: bool = False) -> str:
    if self.success:
      lines = ["[SUCCESS] Evaluación exitosa."]
      lines.append("[SUCCESS] Métricas:")
      for k, v in self.metrics.items():
        lines.append(f"- {k}: {v:.4f}")
      return "\n".join(lines)

    lines = ["[ERROR] Errores durante la evaluación:"]
    for err in self.errors:
      lines.append(f"- {err}")
    if add_warning and self.warnings:
      lines.append("[WARNING] Advertencias:")
      for w in self.warnings:
        lines.append(f"- {w}")
    return "\n".join(lines)

def evaluate_pipeline(pipeline: Pipeline, X: np.ndarray, y: np.ndarray, *, cv: int = 5, scoring: List[str] | None = None) -> EvaluationResult:
  "Evalúa el pipeline con validación cruzada y devuelve métricas."
  errors: List[str] = []
  warnings_list: List[str] = []
  metrics: Dict[str, float] = {}

  if scoring is None:
    scoring = ["accuracy", "f1_weighted", "roc_auc"]
  VALID_SCORING = {"accuracy", "f1_weighted", "roc_auc"}

  # Validaciones iniciales
  if not isinstance(pipeline, Pipeline):
    errors.append("El argumento 'pipeline' no es un sklearn Pipeline.")
    return EvaluationResult(success=False, errors=errors)
  if X is None or y is None:
    errors.append("X e y no pueden ser None.")
    return EvaluationResult(success=False, errors=errors)

  try:
    X = np.asarray(X)
    y = np.asarray(y)
  except Exception as e:
    errors.append(f"No se pudo convertir X o y a numpy array: {e}")
    return EvaluationResult(success=False, errors=errors)

  if X.ndim != 2:
    errors.append(f"X debe ser 2D, se encontró {X.ndim}D.")
  if y.ndim != 1:
    errors.append(f"y debe ser 1D, se encontró {y.ndim}D.")
  if X.shape[0] != y.shape[0]:
    errors.append(f"X e y tienen distinto número de muestras: {X.shape[0]} vs {y.shape[0]}.")
  if X.shape[0] < cv * 2:
    errors.append(f"Pocas muestras ({X.shape[0]}) para {cv} folds. Se necesitan al menos {cv * 2}.")
  unknown_scoring = set(scoring) - VALID_SCORING
  if unknown_scoring:
    errors.append(f"Métricas no reconocidas: {unknown_scoring}. Válidas: {VALID_SCORING}.")

  if errors:
    return EvaluationResult(success=False, errors=errors)

  # Validación particular para ColumnTransformer
  from sklearn.compose import ColumnTransformer
  for step_name, est in pipeline.steps:
    if isinstance(est, ColumnTransformer):
      for name, trans, columns in est.transformers:
        if isinstance(columns, list) and all(isinstance(c, int) for c in columns):
          max_idx = max(columns) if columns else -1
          if max_idx >= X.shape[1]:
            errors.append(f"ColumnTransformer '{step_name}' tiene índices de columna fuera de rango. El índice máximo es {max_idx}, pero X solo tiene {X.shape[1]} columnas.")
  if errors:
    return EvaluationResult(success=False, errors=errors)

  # Ajustes automáticos de parámetros que dependen de X
  from sklearn.feature_selection import SelectKBest, SelectPercentile
  from sklearn.decomposition import PCA

  for step_name, est in pipeline.steps:
    if isinstance(est, (SelectKBest, SelectPercentile)):
      k = est.k
      if isinstance(k, int) and k > X.shape[1]:
        warnings_list.append(f"{type(est).__name__}(k={k}) pero X solo tiene {X.shape[1]} características. Se ajusta k a {X.shape[1]}.")
        est.k = X.shape[1]
    if isinstance(est, PCA):
      n = est.n_components
      if isinstance(n, int) and n > min(X.shape):
        warnings_list.append(f"PCA(n_components={n}) excede min(n_samples, n_features)={min(X.shape)}. Se ajusta a {min(X.shape) - 1}.")
        est.n_components = min(X.shape) - 1

  # Cross‑validation
  n_classes = len(np.unique(y))
  for metric in scoring:
    sklearn_metric = metric
    if metric == "roc_auc" and n_classes > 2:
      sklearn_metric = "roc_auc_ovr"
      warnings_list.append("roc_auc con >2 clases: se utiliza 'roc_auc_ovr'.")
    try:
      scores = cross_val_score(pipeline, X, y, cv=cv, scoring=sklearn_metric)
      metrics[f"{metric}_mean"] = float(np.mean(scores))
      metrics[f"{metric}_std"] = float(np.std(scores))
    except Exception as exc:
      core = _extract_core_error(exc)
      # Mejora: añadir aclaración si el error es por strings en columnas
      if "Specifying the columns using strings" in str(exc):
        core += " (Nota: los datos se pasan como array NumPy; utiliza índices enteros en el campo 'columns'.)"
      errors.append(f"Error en '{metric}': {core}")

  if errors:
    return EvaluationResult(success=False, errors=errors, warnings=warnings_list)

  # Fit completo
  try:
    pipeline.fit(X, y)
    y_pred = pipeline.predict(X)
    metrics["train_accuracy"] = float(accuracy_score(y, y_pred))
  except Exception as exc:
    core = _extract_core_error(exc)
    errors.append(f"Error en fit/predict final: {core}")
    return EvaluationResult(success=False, metrics=metrics, errors=errors, warnings=warnings_list)

  return EvaluationResult(success=True, metrics=metrics, errors=[], warnings=warnings_list)

## Orquestación Principal: Parsear $\to$ Construir $\to$ Evaluar

In [ ]:
@dataclass
class ParseResult:
  "Resultado del parsing de una respuesta del LLM."
  success: bool
  spec: Optional[PipelineSpec] = None
  pipeline: Optional[Pipeline] = None
  errors: List[str] = field(default_factory=list)
  warnings: List[str] = field(default_factory=list)

  def to_feedback(self) -> str:
    if self.success:
      return "[SUCCESS] Pipeline parseado y construido correctamente."
    lines = ["[ERROR] Falló el parsing/construcción del pipeline."]
    for e in self.errors:
      lines.append(f"- {e}")
    if self.warnings:
      lines.append("[WARNING] Advertencias:")
      for w in self.warnings:
        lines.append(f"- {w}")
    return "\n".join(lines)


def parse_and_build(llm_response: str) -> ParseResult:
  "Extrae todos los objetos JSON, intenta validarlos como PipelineSpec, construye el primer pipeline exitoso. Devuelve errores acumulados"
  candidates = extract_json_candidates(llm_response)
  if not candidates:
    return ParseResult(success=False, errors=["No se encontró ningún bloque JSON válido en la respuesta."])

  all_errors: List[str] = []
  warnings_list: List[str] = []

  for idx, candidate in enumerate(candidates, start=1):
    # Verificar que contenga clave "steps"
    if not isinstance(candidate, dict) or "steps" not in candidate:
      all_errors.append(f"Candidato {idx}: falta la clave 'steps' o no es un objeto.")
      continue

    try:
      # Validación con Pydantic
      spec = PipelineSpec.model_validate(candidate)
    except ValidationError as ve:
      # Formatear los errores de Pydantic de manera legible
      error_messages = []
      for err in ve.errors():
        loc = " → ".join(str(x) for x in err["loc"])
        msg = err["msg"]
        error_messages.append(f"  en {loc}: {msg}")
      all_errors.append(f"Candidato {idx} no válido:\n" + "\n".join(error_messages))
      continue

    # Intentar construir el pipeline de sklearn
    try:
      sk_pipeline = build_pipeline(spec)
    except Exception as e:
      all_errors.append(f"Candidato {idx}: la especificación es válida pero falló la construcción: {e}")
      continue

    # Éxito
    return ParseResult(
        success=True,
        spec=spec,
        pipeline=sk_pipeline,
        errors=[],
        warnings=[],
    )

  # Si llegamos aquí, todos los intentos fallaron.
  # Agregamos un mensaje resumen con los fallos acumulados
  feedback = "Ningún candidato pudo convertirse en un pipeline válido.\n"
  feedback += "Fallos detectados:\n" + "\n".join(all_errors)
  return ParseResult(success=False, errors=[feedback], warnings=warnings_list)

## Casos de Prueba con lista de respuestas simuladas  

In [ ]:
from sklearn.datasets import make_classification

# Generar datos de prueba
X, y = make_classification(n_samples=300, n_features=20)
print(f"Datos: X={X.shape}, y={y.shape}")

# Lista de respuestas simuladas (como las que podría devolver un LLM)
test_responses = [
    # 1. Respuesta perfecta
    """
    {
      "steps": [
        {
          "name": "scaler",
          "component": "StandardScaler",
          "hyperparameters": {"with_mean": true, "with_std": true}
        },
        {
          "name": "fs",
          "component": "SelectKBest",
          "hyperparameters": {"k": 10}
        },
        {
          "name": "clf",
          "component": "RandomForestClassifier",
          "hyperparameters": {"n_estimators": 100, "max_depth": 5}
        }
      ]
    }
    """,
    # 2. Hiperparámetro fuera de rango (C de LogisticRegression)
    """
    {
      "steps": [
        {"name": "scaler", "component": "StandardScaler", "hyperparameters": {}},
        {
          "name": "clf",
          "component": "LogisticRegression",
          "hyperparameters": {"C": 2000, "max_iter": 100}
        }
      ]
    }
    """,
    # 3. Componente no permitido
    """
    {
      "steps": [
        {"name": "bad", "component": "XGBClassifier", "hyperparameters": {}}
      ]
    }
    """,
    # 4. ColumnTransformer con sub‑transformador válido
    """
    {
      "steps": [
        {
          "name": "col_trans",
          "component": "ColumnTransformer",
          "hyperparameters": {
            "transformers": [
              {
                "name": "num",
                "transformer": "StandardScaler",
                "columns": [0, 1, 2],
                "transformer_hyperparameters": {"with_mean": true}
              }
            ],
            "remainder": "drop"
          }
        },
        {
          "name": "clf",
          "component": "KNeighborsClassifier",
          "hyperparameters": {"n_neighbors": 5}
        }
      ]
    }
    """,
    # 5. Varios pipelines en un mismo texto (debe extraer el primero bueno)
    """
    Aquí hay un pipeline inicial que falla por un componente incorrecto:
    {"steps":[{"name":"a","component":"FakeComponent","hyperparameters":{}}]}
    Pero luego hay uno correcto:
    ```json
    {
      "steps": [
        {
          "name": "scaler",
          "component": "StandardScaler",
          "hyperparameters": {"with_mean": true, "with_std": false}
        },
        {
          "name": "clf",
          "component": "DecisionTreeClassifier",
          "hyperparameters": {"max_depth": 3}
        }
      ]
    }
    ```
    """,
    # 6. JSON malformado (cierre incorrecto) y luego uno bueno en mismo texto
    """
    Primer intento (roto):
    {"steps": [{"name": "x", "component": "PCA", "hyperparameters": {"n_components": 5}]
    Pero el siguiente sí es correcto:
    {
      "steps": [
        {"name": "clf", "component": "GaussianNB", "hyperparameters": {}}
      ]
    }
    """,
    # 7. Hiperparámetro con tipo incorrecto (string en lugar de int)
    """
    {
      "steps": [
        {
          "name": "fs",
          "component": "SelectKBest",
          "hyperparameters": {"k": "muchos"}
        }
      ]
    }
    """,
    # 8. Pipeline vacío (sin steps)
    """
    {"steps": []}
    """,
    # 9. Texto sin ningún JSON
    "No se me ocurre ningún pipeline hoy, disculpa.",
    # 10. Múltiples bloques de JSON malformados (falta una comilla, etc.) y ninguno válido
    """ 
    Intento 1: {"steps": [{"name": "sc", "component": "StandardScaler", "hyperparameters": {"with_mean": true, "with_std": true}}] 
    Intento 2: {"steps": [{"name": "clf", "component": "LogisticRegression", "hyperparameters": {"C": 0.1 "max_iter": 100}}]}
    """,
    # 11. JSON correcto pero dentro de un comentario más grande y con caracteres extra que lo invalidan
    """
    Aquí tienes un pipeline: steps: [{ "name": "scaler", "component": "StandardScaler" }]. Pero no está completo
    """,
    # 12. Especificación válida según Pydantic, pero falla al construir porque se pasa un valor no soportado por el constructor de sklearn
    """ 
    {
      "steps": [
        {
          "name": "clf",
          "component": "LogisticRegression",
          "hyperparameters": {"C": 1.0, "solver": "invalid_solver", "max_iter": 100}
        }
      ]
    }
    """, 
    # 13. ColumnTransform con columnas que no existen
    """ 
    {
      "steps": [
        {
          "name": "col_trans",
          "component": "ColumnTransformer",
          "hyperparameters": {
            "transformers": [
              {
                "name": "scaler",
                "transformer": "StandardScaler",
                "columns": [50, 51, 52],   # índices fuera de rango (X tiene 20 columnas)
                "transformer_hyperparameters": {}
              }
            ],
            "remainder": "drop"
          }
        },
        {
          "name": "clf",
          "component": "LogisticRegression",
          "hyperparameters": {}
        }
      ]
    }
    """,
    # Error: cross_val_score (sklearn internamente espera un clasificador)
    """ 
    {
      "steps": [
        {
          "name": "scaler",
          "component": "StandardScaler",
          "hyperparameters": {"with_mean": true}
        },
        {
          "name": "fs",
          "component": "SelectKBest",
          "hyperparameters": {"k": 10}
        }
      ]
    }
    """, 
    # Pipeline con un clasificador que requiere que todas las muestras tengan pesos positivos. Error en fit.
    """ 
    {
      "steps": [
        {
          "name": "scaler",
          "component": "StandardScaler",
          "hyperparameters": {}
        },
        {
          "name": "clf",
          "component": "MultinomialNB",
          "hyperparameters": {}
        }
      ]
    }
    """,
    #  
    """ 
    {
      "steps": [
        {
          "name": "scaler",
          "component": "StandardScaler",
          "hyperparameters": {}
        },
        {
          "name": "clf",
          "component": "SVC",
          "hyperparameters": {"kernel": "rbf"}
        }
      ]
    }
    """,
    #  
    """ 
    {
      "steps": [
        {
          "name": "imp",
          "component": "SimpleImputer",
          "hyperparameters": {"strategy": "constant"}
        },
        {
          "name": "clf",
          "component": "LogisticRegression",
          "hyperparameters": {}
        }
      ]
    }
    """ 
]

# Ejecutamos el flujo completo para cada respuesta
for i, response in enumerate(test_responses, start=1):
  print(f"\n{'=' * 60}")
  print(f"📨 Test {i}:")
  print(f"{response}")
  print("-" * 60)
  parse_result = parse_and_build(response)
  print(parse_result.to_feedback())
  if parse_result.success and parse_result.pipeline is not None:
    eval_result = evaluate_pipeline(parse_result.pipeline, X, y)
    print(eval_result.to_feedback(add_warning=False))
  else:
    print("No se pudo construir el pipeline; se omite la evaluación.")

In [ ]:
import sys
sys.path.append('..')  # si chatbot.py está en ../src/ o similar

import numpy as np
from sklearn.pipeline import Pipeline
from src.openml_manager import OpenMLManager
from src.terminal_tools import header, ok, fail, warn
from src.chatbot import Ollama_ChatBot

EXAMPLE = """ 
{
  "steps": [
    {
      "name": "scaler",
      "component": "StandardScaler",
      "hyperparameters": {"with_mean": true, "with_std": true}
    },
    {
      "name": "fs",
      "component": "SelectKBest",
      "hyperparameters": {"k": 10}
    },
    {
      "name": "clf",
      "component": "RandomForestClassifier",
      "hyperparameters": {"n_estimators": 100, "max_depth": 5}
    }
  ]
}
""" 

class AutoML_Bot_Compact(Ollama_ChatBot):
  "AutoML con parsing robusto"

  def __init__(self, model=None, host=None, stream=True, task_description="classification", cv_folds=5, verbose=False):
    self.task_description = task_description
    self.cv_folds = cv_folds
    self.verbose = verbose
    self.dataset = None
    self.dataset_info = None
    self.target_column = None

    # Prompt del sistema mínimo (puedes personalizarlo)
    system_prompt = (
        f"Componentes Permitidos: {ALLOWED_COMPONENTS}"
        "Eres un experto en ML. Diseña un pipeline de clasificación de sklearn en JSON. "
        "Solo usa componentes permitidos y responde ÚNICAMENTE con un JSON válido."
        "Para ColumnTransformer, cada transformador debe ser un diccionario con las claves 'name', 'transformer' (nombre del componente, p.ej. 'StandardScaler'), "
        "'columns' (lista de índices enteros o nombres de columna), y opcionalmente 'transformer_hyperparameters'.\n"
        f"Ejemplo de JSON válido: {EXAMPLE}"
    )
    super().__init__(model=model, host=host, system_prompt=system_prompt, stream=stream)

  def load_dataset_from_openml(self, dataset_id: int) -> bool:
    manager = OpenMLManager(path="./datasets")
    if self.verbose:
      header(f"[CARGA] OpenML dataset {dataset_id}")
    self.dataset = manager.get_dataset(dataset_id=dataset_id)
    self.dataset_info = manager.get_dataset_info(dataset_id=dataset_id)
    self.target_column = self.dataset_info['target']
    if self.verbose:
      ok(f"Dataset cargado: {self.dataset.shape[0]} filas, {self.dataset.shape[1]} columnas, target='{self.target_column}'")
    return True

  def _build_dataset_description(self) -> str:
    "Breve descripción textual para el prompt"
    if self.dataset is None:
      return ""
    cols = list(self.dataset.columns)
    target_col = self.target_column
    feature_cols = [c for c in cols if c != target_col]
    n_samples, n_features = self.dataset.shape[0], len(feature_cols)
    return (
        f"Datos: {n_samples} muestras, {n_features} características.\n"
        f"Columna objetivo: '{target_col}'.\n"
        f"Columnas: {', '.join(cols)}\n"
        f"Primeras 3 filas:\n{self.dataset.head(3).to_string()}\n"
    )

  def _user_prompt(self, error_msg: str = "") -> str:
    base = (
      f"{self._build_dataset_description()}\n"
      "Genera un pipeline de clasificación para este dataset.\n"
      "Los datos se proporcionarán como array NumPy, por lo que si usas ColumnTransformer, "
      "los índices de columna deben ser números enteros (no nombres)."
    )
    if error_msg:
      print(f"Error que se proporciona al LLM: {error_msg}")
      print("=" * 50)
      base += f"\n\nEl pipeline anterior falló con estos errores:\n{error_msg}\nCorrígelos y responde con SOLO un JSON válido (sin bloques de código markdown)."
    return base

  def generate_pipeline(self, max_attempts: int = 3) -> tuple[Pipeline, Dict] | None:
    "Genera un pipeline usando parse_and_build y evaluate_pipeline"
    if self.dataset is None:
      print("Error: Dataset no cargado.")
      return None

    X = self.dataset.drop(columns=[self.target_column]).to_numpy()
    y = self.dataset[self.target_column].to_numpy()
    prompt = self._user_prompt()

    for attempt in range(1, max_attempts + 1):
      header(f"[Intento {attempt}/{max_attempts}]")
      response = self.chat(prompt)
      raw_text = response.message.content

      # Parseo robusto (usa parse_and_build del notebook)
      parse_result = parse_and_build(raw_text)
      if not parse_result.success:
        error_feedback = parse_result.to_feedback()
        fail(f"Parseo falló:\n{error_feedback}")
        prompt = self._user_prompt(error_feedback)
        continue

      pipeline = parse_result.pipeline
      ok("Pipeline construido exitosamente.")

      # Evaluación (usa evaluate_pipeline del notebook)
      eval_result = evaluate_pipeline(pipeline, X, y, cv=self.cv_folds, scoring=["accuracy"])
      if not eval_result.success:
        error_feedback = eval_result.to_feedback()
        fail(f"Evaluación falló:\n{error_feedback}")
        prompt = self._user_prompt(error_feedback)
        continue

      ok(f"Métricas: {eval_result.metrics}")
      return pipeline, eval_result.metrics

    print("No se logró generar un pipeline válido.")
    return None

Modelos disponibles: `qwen3-coder:480b`, `gpt-oss:120b`, `gpt-oss:20b`

In [ ]:
# Crear instancia
bot = AutoML_Bot_Compact(
  verbose=False, 
  cv_folds=5
)

# Cargar dataset y generar pipeline
bot.load_dataset_from_openml(dataset_id=31)  # dataset credit-g (o el que prefieras)

result = bot.generate_pipeline(max_attempts=5, )

if result:
  pipeline, metrics = result
  print("\nPipeline final:")
  print(pipeline)
  print("Métricas:", metrics)

In [ ]:
for i, msg in enumerate(bot._messages, 1):
  print(f"{i}. {msg.role}")
  print(msg.content)